# Apriori — Member Association Rules

Mencari association rules (produk yang sering dibeli bersamaan) khusus untuk transaksi **Member**.

**Pipeline:**
1. Load basket data & filter member transactions
2. Join segment labels dari K-Means member
3. Run Apriori → frequent itemsets
4. Generate association rules (lift, confidence)
5. Simpan hasil untuk app

## Import Library

In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import gc

from mlxtend.frequent_patterns import apriori, association_rules

import warnings
warnings.filterwarnings('ignore')

print('Library imported.')

Library imported.


## Load Basket Data

Menggunakan PyArrow langsung untuk menghindari error categorical dtype.

In [2]:
gc.collect()

# Read basket via PyArrow
basket_table = pq.read_table('df_basket_apriori.parquet')
print(f'Basket table: {basket_table.num_rows:,} rows x {basket_table.num_columns} cols')
print('Schema:', basket_table.schema)

gc.collect()

Basket table: 9,064,669 rows x 9 cols
Schema: Americano: int8
Cappuccino: int8
Espresso: int8
Flat White: int8
Hot Chocolate: int8
Latte: int8
Matcha Latte: int8
Mocha: int8
transaction_id: large_string
-- schema metadata --
pandas: '{"index_columns": ["transaction_id"], "column_indexes": [{"name"' + 1290


0

## Join dengan Member Status

Ambil `member_status` dari `df_transaction_features.parquet` untuk filter transaksi member saja.

In [3]:
gc.collect()

# Read transaction features (hanya kolom yang diperlukan)
tx_table = pq.read_table(
    'df_transaction_features.parquet',
    columns=['transaction_id', 'user_id', 'member_status']
)
print(f'TX features: {tx_table.num_rows:,} rows')

gc.collect()

TX features: 14,623,691 rows


0

### Join Basket + Transaction Features via PyArrow

In [4]:
gc.collect()

# Join basket dengan member_status
# PyArrow join on transaction_id
join_table = basket_table.join(
    tx_table,
    keys='transaction_id',
    join_type='inner'
)

print(f'Joined table: {join_table.num_rows:,} rows')
gc.collect()

# Filter hanya MEMBER
import pyarrow.compute as pc
mask_member = pc.equal(join_table.column('member_status'), 'Member')
member_table = join_table.filter(mask_member)
print(f'Member transactions: {member_table.num_rows:,} rows')

# Drop member_status & user_id column (bukan produk)
member_table = member_table.drop(['member_status', 'user_id'])

gc.collect()

Joined table: 9,064,669 rows
Member transactions: 4,532,595 rows


0

### Konversi ke Pandas DataFrame (untuk mlxtend)

In [5]:
gc.collect()

# Konversi ke pandas
member_pd = member_table.to_pandas()

# Set transaction_id sebagai index
member_pd = member_pd.set_index('transaction_id')

# Cast ke bool (0/1)
member_pd = member_pd.astype(bool)

print(f'Member basket shape: {member_pd.shape}')
print(f'Member unique transactions: {member_pd.index.nunique():,}')
print()
print('Sample (5 transaksi pertama):')
print(member_pd.head(5))

del join_table, tx_table, basket_table
gc.collect()

Member basket shape: (4532595, 8)
Member unique transactions: 4,532,595

Sample (5 transaksi pertama):
                                      Americano  Cappuccino  Espresso  \
transaction_id                                                          
0b19f01b-20ab-469a-8801-0bdf3db6ccbf       True        True     False   
0b19f0d6-f771-4254-ad92-66b7d0f4332e      False       False      True   
0b19f3dd-99b0-408c-ae53-fd07d72f9457       True        True     False   
0b19f5f0-8486-4f21-9ef1-bbab2d8bfd74       True        True     False   
0b19f883-b46d-44b2-a1ef-f99ac5dcd252       True       False     False   

                                      Flat White  Hot Chocolate  Latte  \
transaction_id                                                           
0b19f01b-20ab-469a-8801-0bdf3db6ccbf        True          False  False   
0b19f0d6-f771-4254-ad92-66b7d0f4332e       False          False  False   
0b19f3dd-99b0-408c-ae53-fd07d72f9457       False          False  False   
0b19f5f0-8486-4

0

## Run Apriori — Frequent Itemsets

Mencari itemset yang sering muncul bersama dalam transaksi member.

In [6]:
min_support = 0.01
print(f'Running Apriori with min_support={min_support}...')
gc.collect()

frequent_itemsets = apriori(
    member_pd,
    min_support=min_support,
    use_colnames=True,
    low_memory=True
)

print(f'Frequent itemsets found: {len(frequent_itemsets):,}')
print(frequent_itemsets.head(10))

Running Apriori with min_support=0.01...
Frequent itemsets found: 36
    support                            itemsets
0  0.294440              frozenset({Americano})
1  0.294241             frozenset({Cappuccino})
2  0.293977               frozenset({Espresso})
3  0.293988             frozenset({Flat White})
4  0.293813          frozenset({Hot Chocolate})
5  0.294115                  frozenset({Latte})
6  0.294131           frozenset({Matcha Latte})
7  0.294002                  frozenset({Mocha})
8  0.061016  frozenset({Cappuccino, Americano})
9  0.060826    frozenset({Espresso, Americano})


## Generate Association Rules

Filter berdasarkan lift > 1 (positive association) dan confidence minimal.

In [7]:
min_threshold = 0    # semua rules (filter lift/confidence via app)
print(f'Generating association rules with min_threshold={min_threshold}...')
gc.collect()

rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=min_threshold
)

# Sort by lift descending
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Association rules found: {len(rules):,}')
print()
print('Top 10 rules by lift:')
display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
print(rules[display_cols].head(10))

Generating association rules with min_threshold=0...
Association rules found: 56

Top 10 rules by lift:
                  antecedents                 consequents   support  \
0          frozenset({Mocha})   frozenset({Matcha Latte})  0.061129   
1   frozenset({Matcha Latte})          frozenset({Mocha})  0.061129   
2     frozenset({Cappuccino})  frozenset({Hot Chocolate})  0.061107   
3  frozenset({Hot Chocolate})     frozenset({Cappuccino})  0.061107   
4      frozenset({Americano})          frozenset({Latte})  0.061165   
5          frozenset({Latte})      frozenset({Americano})  0.061165   
6     frozenset({Cappuccino})          frozenset({Latte})  0.061105   
7          frozenset({Latte})     frozenset({Cappuccino})  0.061105   
8       frozenset({Espresso})  frozenset({Hot Chocolate})  0.060981   
9  frozenset({Hot Chocolate})       frozenset({Espresso})  0.060981   

   confidence      lift  
0    0.207920  0.706898  
1    0.207829  0.706898  
2    0.207677  0.706835  
3    0.207

In [8]:
# Summary of rules
print('=== Rules Summary ===')
print(f'Total rules: {len(rules):,}')
print(f'Avg lift: {rules["lift"].mean():.3f}')
print(f'Avg confidence: {rules["confidence"].mean():.3f}')
print(f'Avg support: {rules["support"].mean():.4f}')
print()
print(f'Rules with lift > 2: {len(rules[rules["lift"] > 2]):,}')
print(f'Rules with lift > 3: {len(rules[rules["lift"] > 3]):,}')
print(f'Rules with lift > 5: {len(rules[rules["lift"] > 5]):,}')

=== Rules Summary ===
Total rules: 56
Avg lift: 0.704
Avg confidence: 0.207
Avg support: 0.0609

Rules with lift > 2: 0
Rules with lift > 3: 0
Rules with lift > 5: 0


## Save Results

Menyimpan rules untuk digunakan di app.

In [9]:
gc.collect()

# Konversi frozenset ke string agar bisa disimpan
rules_out = rules.copy()
rules_out['antecedents'] = rules_out['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
rules_out['consequents'] = rules_out['consequents'].apply(lambda x: ', '.join(sorted(list(x))))

# Save parquet
rules_out.to_parquet('df_rules_member.parquet', index=False)
print('Saved: df_rules_member.parquet')

# Save CSV (view-friendly)
rules_out.to_csv('df_rules_member.csv', index=False)
print('Saved: df_rules_member.csv')

print(f'\nDone. {len(rules_out):,} association rules saved.')

Saved: df_rules_member.parquet
Saved: df_rules_member.csv

Done. 56 association rules saved.
